# Marvin smoke test (SDSS-IV MaNGA)

## Order matters on Windows

1. **Kernel → Restart** (if you ever imported Marvin before fixing logging).
2. Run **the first code cell** (NTFS log fix) before anything else.
3. Then `%pip` / `import marvin`.

If `%pip` runs while Marvin is already loaded, pip’s `ResourceWarning`s can be routed through Marvin’s file logger and trigger **log rotation** using a filename like `...2025-05-24_00:00:00` — the **colons are illegal on NTFS**, hence `WinError 123`.

## Dependencies

Pin **marshmallow 3.x** (Marshmallow 4 removed `__version__` and breaks older `webargs`):

```bash
python -m pip install "marshmallow>=3.21,<4"
```

**Docs:** [Marvin](https://sdss-marvin.readthedocs.io/en/stable/), [Maps](https://sdss-marvin.readthedocs.io/en/stable/tools/maps.html).

In [1]:
"""
Run this first after a kernel restart.
Replaces ':' in rotated log paths (e.g. ...T00:00:00) — invalid on Windows except for 'C:'.
"""
import logging.handlers
import os


def _apply_ntfs_timed_rotating_logfix():
    if os.name != "nt":
        return
    cls = logging.handlers.TimedRotatingFileHandler
    if getattr(cls, "_ntfs_rot_fix", False):
        return

    _orig_rf = cls.rotation_filename

    def rotation_filename(self, default_name):
        path = _orig_rf(self, default_name)
        drive, rest = os.path.splitdrive(path)
        if ":" in rest:
            path = drive + rest.replace(":", "-")
        return path

    cls.rotation_filename = rotation_filename  # type: ignore[assignment]
    cls._ntfs_rot_fix = True
    print("TimedRotatingFileHandler: NTFS-safe rotation_filename patch applied")


_apply_ntfs_timed_rotating_logfix()

TimedRotatingFileHandler: NTFS-safe rotation_filename patch applied


In [2]:
# Optional: install in terminal instead of here to avoid extra warnings during pip.
%pip install -q "marshmallow>=3.21,<4"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import marvin
from importlib.metadata import version, PackageNotFoundError

try:
    print("sdss-marvin:", version("sdss-marvin"))
except PackageNotFoundError:
    pass
print("marvin:", getattr(marvin, "__version__", "n/a"))
try:
    print("marshmallow:", version("marshmallow"))
except PackageNotFoundError:
    pass

[WARNING]: cannot initiate Sentry error reporting: unknown error. (UserWarning)
[INFO]: No release version set. Setting default to DR17
[WARNING]: path C:\Users\james\sas\dr17\manga\spectro\redux\v3_1_1\drpall-v3_1_1.fits cannot be found. Setting drpall to None. (MarvinUserWarning)
[WARNING]: path C:\Users\james\sas\dr17\manga\spectro\analysis\v3_1_1\3.1.0\dapall-v3_1_1-3.1.0.fits cannot be found. Setting dapall to None. (MarvinUserWarning)
[WARNING]: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html (TqdmWarning)


sdss-marvin: 2.8.2
marvin: 2.8.2
marshmallow: 3.26.2


In [4]:
from marvin.tools import Maps

PLATE_IFU = "8485-1901"

maps = Maps(PLATE_IFU)
print(maps)
print("mangaid:", getattr(maps, "mangaid", None), "plateifu:", getattr(maps, "plateifu", None))

if getattr(maps, "dapall", None) is not None:
    try:
        print("dapall sfr_tot:", maps.dapall["sfr_tot"])
    except (KeyError, TypeError, IndexError):
        print("dapall keys sample:", list(maps.dapall)[:12], "...")

if getattr(maps, "nsa", None) is not None:
    try:
        print("NSA z:", maps.nsa["z"])
    except (KeyError, TypeError, IndexError):
        print("NSA:", maps.nsa)

[WARNING]: Cannot retrieve URLMap. Remote functionality will not work: Requests Http Status Error: 504 Server Error: Gateway Time-out for url: https://magrathea.sdss.org/marvin/api/general/getroutemap?release=DR17&compression=json (MarvinUserWarning)
[ERROR]: Traceback (most recent call last):
  File "a:\Python\envs\MU\lib\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\james\AppData\Local\Temp\ipykernel_42200\1908027778.py", line 5, in <module>
    maps = Maps(PLATE_IFU)
  File "a:\Python\envs\MU\lib\site-packages\marvin\tools\maps.py", line 113, in __init__
    self._load_maps_from_api()
  File "a:\Python\envs\MU\lib\site-packages\marvin\tools\maps.py", line 371, in _load_maps_from_api
    url = marvin.config.urlmap['api']['getMaps']['url']
  File "a:\Python\envs\MU\lib\site-packages\brain\core\core.py", line 37, in __missing__
    raise BrainError('No URL Map found. Cannot make remote call

In [ ]:
dm = maps.datamodel
prop_list = list(dm.properties) if hasattr(dm, "properties") else list(dm)


def _prop_key(p):
    for attr in ("name", "full", "fullname", "full_name"):
        if hasattr(p, attr):
            v = getattr(p, attr)
            if isinstance(v, str) and v:
                return v
    return str(p)


names = sorted({_prop_key(p) for p in prop_list})
print(f"Datamodel entries: {len(names)}")
print("Sample:", names[:30])
hits = [n for n in names if any(s in n.lower() for s in ("sfr", "sfh", "star", "form"))]
print("Hits (sfr|sfh|star|form):", hits[:40])

In [ ]:
import numpy as np

ha = maps["emline_gflux_ha_6564"]
print(type(ha))
arr = np.asarray(ha.value)
print("Hα shape:", arr.shape, arr.dtype)
if isinstance(ha.value, np.ma.MaskedArray):
    print("masked fraction:", float(np.mean(ha.value.mask)))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
ha.plot(fig=fig, ax=ax)
ax.set_title(f"Hα — {PLATE_IFU}")
plt.tight_layout()
plt.show()

`ResourceWarning: unclosed file` from pip/IPython is annoying but harmless; the **WinError 123** was from **colons in the rotated log filename** — fixed by the first cell **if** it runs before Marvin attaches its file handler. **Restart kernel**, then run top-to-bottom.